## INM713 Semantic Web Technologies and Knowledge Graphs 

### Coursework Project (Part 2): Ontology Alignment (Task OA)

In [1]:
# Matthew Lau (250060300)

### Deafult Setting

In [2]:
# System name: Matthew Lau Ontology Aligner (MLOA)
# To produce alignments with two different ontologies in Turtle format by combing lexical, structural and LLM-based techniques

In [3]:
# Install libraries
!pip install owlready2==0.45
!pip install rdflib==7.1.3 
!pip install python-Levenshtein==0.26.1 
!pip install owlrl==7.1.3 
!pip install pandas==2.2.3 
!pip install sentence-transformers

In [4]:
# Import Libraries
from rdflib import Graph, URIRef, Namespace
from rdflib.namespace import OWL, RDF, RDFS

# Source: Laboratory 5 support code by Ernesto Jiménez-Ruiz
# Repository:https://github.com/city-knowledge-graphs/python-2026/tree/f3a1ce17323e95a45d253b1a5fcc1ad4a3fa806d/lab5
import Levenshtein as lev
from isub import isub

import time
import os
import pandas as pd
import requests
import json

In [5]:
# Base path for all OAEI datasets
OAEI_BASE = os.path.expanduser("~/Desktop/OAEI-INM713-IN3067")

# Conference dataset paths
CONFERENCE_ONTO_PATH = os.path.join(OAEI_BASE, "conference", "ontologies")
CONFERENCE_REF_PATH = os.path.join(OAEI_BASE, "conference")

In [6]:
# Loading Ontologies and Extracting Labels
def getLabel(entity):
    " Extract the best available label for an ontology entity."
    if len(entity.label) > 0:
        return str(entity.label[0]) # Use rdfs:label if available
    else:
        return str(entity.name) # Otherwise use URI name
        
def loadOntology(path):
    g = Graph()
    
    if path.endswith(".ttl"):
        g.parse(path, format="turtle")
    else:
        g.parse(path, format="xml") 
    return g
    
def getClasses(g):
    """
    Extract all OWL classes from an ontology graph.
    """
    classes = []
    for s in g.subjects(RDF.type, OWL.Class):
        if isinstance(s, URIRef):  # exclude blank nodes
            classes.append(s)
    return classes
    
def getObjectProperties(g):
    """
    Extract all OWL object properties from an ontology graph.
    """
    props = []
    for s in g.subjects(RDF.type, OWL.ObjectProperty):
        if isinstance(s, URIRef):
            props.append(s)
    return props

def getDataProperties(g):
    """
    Extract all OWL data properties from an ontology graph.
    """
    props = []
    for s in g.subjects(RDF.type, OWL.DatatypeProperty):
        if isinstance(s, URIRef):
            props.append(s)
    return props
    
def getIndividuals(g):
    """
    Extract all OWL named individuals from an ontology graph.
    """
    individuals = []
    for s in g.subjects(RDF.type, OWL.NamedIndividual):
        if isinstance(s, URIRef):
            individuals.append(s)
    return individuals

def getLabel(g, entity):
    """
    Extract the best available label for an ontology entity.
    """
    for label in g.objects(entity, RDFS.label):
        return str(label)
        
    uri = str(entity)
    if "#" in uri:
        return uri.split("#")[-1]   
    else:
        return uri.split("/")[-1] 
        
def getOntologyInfo(g, onto_name="Ontology"):
    """
    Print a summary of an ontology.
    """
    classes = getClasses(g)
    obj_props = getObjectProperties(g)
    data_props = getDataProperties(g)
    individuals = getIndividuals(g)

    print(f"=== {onto_name} Summary ===")
    print(f"Classes:            {len(classes)}")
    print(f"Object properties:  {len(obj_props)}")
    print(f"Data properties:    {len(data_props)}")
    print(f"Individuals:        {len(individuals)}")

    print(f"\nFirst 5 classes and their labels:")
    for cls in classes[:5]:
        print(f"  URI:   {cls}")
        print(f"  Label: {getLabel(g, cls)}")
        print()

In [7]:
# Test the function 
cmt_path = os.path.join(CONFERENCE_ONTO_PATH, "cmt.owl")
ekaw_path = os.path.join(CONFERENCE_ONTO_PATH, "ekaw.owl")

print("Loading ontologies")
g1 = loadOntology(cmt_path)
g2 = loadOntology(ekaw_path)
print(f"{g1,g2}\nBoth ontologies loaded successfully!\n")

getOntologyInfo(g1, "cmt.owl")
print()
getOntologyInfo(g2, "ekaw.owl")

Loading ontologies
(<Graph identifier=Nfdad1d4ccaec4a59b098993c3d382098 (<class 'rdflib.graph.Graph'>)>, <Graph identifier=Nec2ecfe52a2e42a8a19b37009b20eb12 (<class 'rdflib.graph.Graph'>)>)
Both ontologies loaded successfully!

=== cmt.owl Summary ===
Classes:            29
Object properties:  49
Data properties:    10
Individuals:        0

First 5 classes and their labels:
  URI:   http://cmt#Meta-Reviewer
  Label: Meta-Reviewer

  URI:   http://cmt#Reviewer
  Label: Reviewer

  URI:   http://cmt#Decision
  Label: Decision

  URI:   http://cmt#Person
  Label: Person

  URI:   http://cmt#Document
  Label: Document


=== ekaw.owl Summary ===
Classes:            73
Object properties:  33
Data properties:    0
Individuals:        0

First 5 classes and their labels:
  URI:   http://ekaw#Event
  Label: Event

  URI:   http://ekaw#Person
  Label: Person

  URI:   http://ekaw#Accepted_Paper
  Label: Accepted_Paper

  URI:   http://ekaw#Rejected_Paper
  Label: Rejected_Paper

  URI:   http:/

### Subtask OA.1 (Give the system a name and use one base alignment technique)

In [8]:
# Section 1: basic alignment technique - Lexical Matching - Class Matching
# Lexical matching compares the labels of entities from two ontologies using string similarity, such as exacting mathing and Isub similarity.

In [9]:
# Source: "A String Metric for Ontology Alignment" from Australian National University
# Repository:https://sourceforge.net/projects/febrl/

def normaliseLabel(label):
    """
    Normalise a label for comparison:
    1. Convert to lowercase
    2. Remove underscores and hyphens
    3. Strip whitespace

    Example:
        "hasConferenceChair" → "hasconferencechair"
        "conference_chair"   → "conferencechair"
    """
    label = label.lower()
    label = label.replace("_", "").replace("-", "")
    label = label.strip()
    return label

def lexicalSimilarity(label1, label2):
    """
    Compute similarity between two labels.
    Returns a score between 0.0 and 1.0.

    Uses ISub algorithm designed specifically for ontology alignment.
    """
    # Normalise both labels first
    norm1 = normaliseLabel(label1)
    norm2 = normaliseLabel(label2)

    # Exact match → score 1.0
    if norm1 == norm2:
        return 1.0

    # ISub similarity for non-exact matches
    return isub(norm1, norm2)

def lexicalMatcher(g1, g2, threshold=0.9):
    """
    Basic lexical matcher: compare all classes from two ontologies.
    Returns a list of mappings (entity1, entity2, score).

    Parameters:
        g1: source ontology
        g2 : target ontology
        threshold : minimum similarity score to accept a mapping (default 0.9 — only very similar labels)
    """
    mappings = []

    # Get all classes from both ontologies
    classes1 = getClasses(g1)
    classes2 = getClasses(g2)

    print(f"Comparing {len(classes1)} classes "
          f"vs {len(classes2)} classes")
    print(f"Total comparisons: "
          f"{len(classes1) * len(classes2)}\n")

     # Compare every class in g1 against every class in g2
    for c1 in classes1:
        label1 = getLabel(g1, c1)

        for c2 in classes2:
            label2 = getLabel(g2, c2)

            # Calculate similarity
            score = lexicalSimilarity(label1, label2)

            # Only keep mappings above threshold
            if score >= threshold:
                mappings.append((c1, c2, score))

    print(f"Found {len(mappings)} mappings "
          f"above threshold {threshold}")
    return mappings


In [10]:
# Run Lexical Matching on cmt.owl vs ekaw.owl

In [11]:
start_time = time.time()

mappings = lexicalMatcher(g1, g2, threshold=0.9)

end_time = time.time()
elapsed = round(end_time - start_time, 2)

print(f"\nComputation time: {elapsed} seconds")
print(f"\nSample mappings found:")
print(f"{'Source':<40} {'Target':<40} {'Score'}")
print("-" * 85)

for src, tgt, score in mappings[:10]:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<40} {tgt_label:<40} {score:.3f}")

Comparing 29 classes vs 73 classes
Total comparisons: 2117

Found 12 mappings above threshold 0.9

Computation time: 0.16 seconds

Sample mappings found:
Source                                   Target                                   Score
-------------------------------------------------------------------------------------
Reviewer                                 Review                                   0.957
Person                                   Person                                   1.000
Document                                 Document                                 1.000
Conference                               Conference_Paper                         0.940
Conference                               Conference_Session                       0.922
Conference                               Conference_Trip                          0.950
Conference                               Conference                               1.000
Conference                               Conference_Banq

In [12]:
# One-to-One mapping constraint to modify the accuracy for basic Lexical Matching
# It means each entity can only be matched to its best candidate
def lexicalMatcherOneToOne(g1, g2, threshold=0.9):
    """
    Improved lexical matcher with one-to-one mapping constraint.
    Each entity in g1 is matched to at most ONE entity in g2
    (the one with the highest similarity score).
    """
    mappings = []
    classes1 = getClasses(g1)
    classes2 = getClasses(g2)

    print(f"Comparing {len(classes1)} classes "
          f"vs {len(classes2)} classes")

    for c1 in classes1:
        label1  = getLabel(g1, c1)
        best_score = 0.0
        best_match = None

        for c2 in classes2:
            label2 = getLabel(g2, c2)
            score  = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = c2

        if best_match is not None:
            mappings.append((c1, best_match, best_score))

    print(f"Found {len(mappings)} mappings "
          f"above threshold {threshold}")
    return mappings

In [13]:
# Compare basic lexical matching vs one-to-one mapping
print("=== Basic lexical matching  ===")
mappings_original = lexicalMatcher(g1, g2, threshold=0.9)

print("\n=== One-to-One Matcher ===")
mappings_improved = lexicalMatcherOneToOne(g1, g2, threshold=0.9)

print("\n=== Comparison ===")
print(f"Original mappings:  {len(mappings_original)}")
print(f"Improved mappings:  {len(mappings_improved)}")

print(f"\nImproved mappings detail:")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in mappings_improved:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

=== Basic lexical matching  ===
Comparing 29 classes vs 73 classes
Total comparisons: 2117

Found 12 mappings above threshold 0.9

=== One-to-One Matcher ===
Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9

=== Comparison ===
Original mappings:  12
Improved mappings:  8

Improved mappings detail:
Source                              Target                              Score
---------------------------------------------------------------------------
Reviewer                            Review                              0.957
Person                              Person                              1.000
Document                            Document                            1.000
Conference                          Conference                          1.000
ConferenceChair                     Conference                          0.940
ConferenceMember                    Conference                          0.931
Review                              Review              

In [14]:
# Section 2: basic alignment technique - Property Matching

In [15]:
# Extends lexical matching to object and data properties
# Source: Laboratory 5 support code "AccessEntityLabels.pyby Ernesto Jiménez-Ruiz
# Repository:https://github.com/city-knowledge-graphs/python-2026/blob/f3a1ce17323e95a45d253b1a5fcc1ad4a3fa806d/lab5/AccessEntityLabels.py

def propertyMatcher(g1, g2, threshold=0.9):
    """
    Match object properties and data properties between two ontologies using lexical similarity.
    Uses one-to-one mapping constraint: each property matched to its best candidate only.
    """
    # Object Properties 
    obj_props1 = getObjectProperties(g1)
    obj_props2 = getObjectProperties(g2)

    print(f"Object properties: "
          f"{len(obj_props1)} vs {len(obj_props2)}")
    print(f"Total comparisons: "
          f"{len(obj_props1) * len(obj_props2)}")

    obj_mappings = []
    for p1 in obj_props1:
        label1 = getLabel(g1, p1)
        best_score = 0.0
        best_match = None

        for p2 in obj_props2:
            label2 = getLabel(g2, p2)
            score = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = p2

        if best_match is not None:
            obj_mappings.append((p1, best_match, best_score))

    print(f"Object property mappings found: "
          f"{len(obj_mappings)}\n")

    # Data Properties
    data_props1 = getDataProperties(g1)
    data_props2 = getDataProperties(g2)

    print(f"Data properties: "
          f"{len(data_props1)} vs {len(data_props2)}")
    print(f"Total comparisons: "
          f"{len(data_props1) * len(data_props2)}")

    data_mappings = []
    for p1 in data_props1:
        label1     = getLabel(g1, p1)
        best_score = 0.0
        best_match = None

        for p2 in data_props2:
            label2 = getLabel(g2, p2)
            score  = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = p2

        if best_match is not None:
            data_mappings.append((p1, best_match, best_score))

    print(f"Data property mappings found: "
          f"{len(data_mappings)}")

    return obj_mappings, data_mappings


In [16]:
# Run property matcher waon cmt.owl vs ekaw.owl
start_time = time.time()

obj_mappings, data_mappings = propertyMatcher(g1, g2, threshold=0.9)

end_time = time.time()
elapsed  = round(end_time - start_time, 2)

print(f"\nComputation time: {elapsed} seconds")

print(f"\n=== Object Property Mappings ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in obj_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

print(f"\n=== Data Property Mappings ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in data_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 1

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0

Computation time: 0.14 seconds

=== Object Property Mappings ===
Source                              Target                              Score
---------------------------------------------------------------------------
writtenBy                           writtenBy                           1.000

=== Data Property Mappings ===
Source                              Target                              Score
---------------------------------------------------------------------------


In [17]:
# Try different thresholds for property matching

for threshold in [0.9, 0.8, 0.7, 0.6]:
    obj_m, data_m = propertyMatcher(g1, g2, threshold=threshold)
    print(f"Threshold {threshold}: "
          f"{len(obj_m)} object property mappings\n")

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 1

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.9: 1 object property mappings

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.8: 7 object property mappings

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 22

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.7: 22 object property mappings

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 38

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.6: 38 object property mappings



In [18]:
# Test threshold 0.8 property mappings
obj_mappings_08, _ = propertyMatcher(g1, g2, threshold=0.8)

print(f"\n=== Object Property Mappings (threshold=0.8) ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)

for src, tgt, score in obj_mappings_08:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0

=== Object Property Mappings (threshold=0.8) ===
Source                              Target                              Score
---------------------------------------------------------------------------
assignedByReviewer                  hasReviewer                         0.821
assignReviewer                      hasReviewer                         0.883
reviewCriteriaEnteredBy             reviewWrittenBy                     0.871
endReview                           reviewWrittenBy                     0.803
hasAuthor                           authorOf                            0.801
writtenBy                           writtenBy                           1.000
writeReview                         reviewWrittenBy                     0.865


In [19]:
# Base on the above result, set the threshold of classes = 0.9, threshold of properties = 0.8

In [20]:
# Section 3: Saving alignments in Turtle Format
 # owl:equivalentClass (class mappings)
 # owl:equivalentProperty (property mappings)

def saveMappings(class_mappings, obj_mappings, data_mappings, output_path):
    """
    Save all mappings as RDF triples in Turtle format.
    
    Parameters:
        class_mappings  : list of (class1, class2, score)
        obj_mappings    : list of (prop1, prop2, score)
        data_mappings   : list of (prop1, prop2, score)
        output_path     : path to save the .ttl file
    """
    # Create a new RDF graph for the alignment
    alignment = Graph()

    # Bind namespaces for cleaner output
    alignment.bind("owl", OWL)
    alignment.bind("rdf", RDF)

    # Class Mappings
    for c1, c2, score in class_mappings:
        alignment.add(( URIRef(c1), OWL.equivalentClass, URIRef(c2)))
        
    # Object Property Mappings
    for p1, p2, score in obj_mappings:
        alignment.add((URIRef(p1), OWL.equivalentProperty, URIRef(p2)))

    # Data propoerty Mappings
    for p1, p2, score in data_mappings:
        alignment.add((URIRef(p1),OWL.equivalentProperty,URIRef(p2)))
    
    # Saving 
    alignment.serialize(destination=output_path, format="turtle")

    print(f" Alignment saved to: {output_path}")
    print(f" Class mappings:         {len(class_mappings)}")
    print(f" Object prop mappings:   {len(obj_mappings)}")
    print(f" Data prop mappings:     {len(data_mappings)}")
    print(f" Total mappings:         "
          f"{len(class_mappings) + len(obj_mappings) + len(data_mappings)}")

In [21]:
# Save cmt-ekaw alignment
output_dir = os.path.join(os.path.expanduser("~/Desktop"), "MLOA", "Ontology_Alignments", "OA.4")

# Create directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# File name follows coursework convention:
# mysystem-onto1-onto2.ttl
output_file = os.path.join(output_dir, "mloa-cmt-ekaw.ttl")

# Get class mappings (one-to-one, threshold 0.9)
class_mappings = lexicalMatcherOneToOne(g1, g2, threshold=0.9)

# Get property mappings (threshold 0.8)
obj_mappings, data_mappings = propertyMatcher(g1, g2, threshold=0.8)

# Save all mappings
saveMappings(class_mappings, obj_mappings, data_mappings, output_file)

Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9
Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
 Alignment saved to: C:\Users\siuju/Desktop\MLOA\Ontology_Alignments\OA.4\mloa-cmt-ekaw.ttl
 Class mappings:         8
 Object prop mappings:   7
 Data prop mappings:     0
 Total mappings:         15


In [22]:
# Section 4:  Precision (Pr), Recall(Re), and F-score (F) to evaluate an alignment 
# - Precision = correct mappings / total computed mappings
# - Recall    = correct mappings / total reference mappings  
# - F-score   = harmonic mean of Precision and Recall

In [23]:
# Source: Lab 5 support code "oa_notebook.ipynb" by Ernesto Jiménez-Ruiz
# Repository: https://github.com/city-knowledge-graphs/python-2026/blob/f3a1ce17323e95a45d253b1a5fcc1ad4a3fa806d/lab5/oa_notebook.ipynb
def evaluate(reference_file, system_file):
    """
    Compare system alignment against reference alignment.
    Computes Precision, Recall and F-score.
    """
    # Load reference alignment
    ref = Graph()
    ref.parse(reference_file, format="turtle")

    # Load system alignment
    sys_graph = Graph()
    sys_graph.parse(system_file, format="turtle")

    # Calculate TP, FP, FN
    tp = 0  # Correct mappings
    fp = 0  # Wrong mappings in system but not in reference
    fn = 0  # Missed mappings in reference but not in system

    # Check each system mapping against reference
    for s, p, o in sys_graph:
        if (s, p, o) in ref or (o, p, s) in ref:
            tp += 1  # Correct
        else:
            fp += 1  # Incorrect

    # Check each reference mapping against system
    for s, p, o in ref:
        if (s, p, o) not in sys_graph and \
           (o, p, s) not in sys_graph:
            fn += 1  # Missed
            
    # Calculate metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f_score = (2 * precision * recall) / \
                (precision + recall) \
                if (precision + recall) > 0 else 0
    
    print(f"Evaluating: {os.path.basename(system_file)}")
    print(f"Reference:  {os.path.basename(reference_file)}")
    print(f"─" * 45)
    print(f"True Positives  (TP): {tp}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")
    print(f"─" * 45)
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F-score:   {f_score:.3f}")

    return precision, recall, f_score


In [24]:
# Evalute my system alignment vs reference alignment
# Reference alignment path
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment path
system_file = output_file  # mloa-cmt-ekaw.ttl

# Run evaluation
precision, recall, f_score = evaluate(reference_file, system_file)

Evaluating: mloa-cmt-ekaw.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 10
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.333
Recall:    0.455
F-score:   0.385


In [25]:
# Results Tracking Initialisation
results = {"Task": [],"Technique": [],"Precision": [],"Recall": [],"F-score": [],"Time(s)": []}

In [26]:
# Section 5: Result Tracking of my MLOA system
# Record improvement after each techinque used

# Record first result
results["Task"].append("cmt-ekaw")
results["Technique"].append("Lexical (threshold=0.9/0.8)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed, 3))

df_results = pd.DataFrame(results)

# Display as table
print("=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35} "
          f"{row['Precision']:>10.3f} {row['Recall']:>8.3f} "
          f"{row['F-score']:>8.3f} {row['Time(s)']:>8.2f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385     0.14


### Subtask OA.1 (Combine one or more techniques)

In [27]:
# Section 1: Combined alignment technique - Structural Matching 
# Extend lexical matching with structural information. Use parent class relationships to improve class mapping. 
# For example, if two classess have smiliar names and both have smiliar parent classes, then the mapping is considered more reliable.

In [28]:
def getParentClasses(g, entity):
    """
    Get all parent classes of an entity via rdfs:subClassOf.
    """
    parents = []
    for parent in g.objects(entity, RDFS.subClassOf):
        if isinstance(parent, URIRef):
            parents.append(parent)
    return parents

def structuralScore(g1, g2, c1, c2, lexical_mappings):
    """
    Calculate structural similarity between two classes.
    Logic:
    - Get parent classes of c1 and c2
    - Check if any parent pairs are already matched in lexical_mappings

    Parameters:
        g1: rdflib Graph (source ontology)
        g2: rdflib Graph (target ontology)
        c1: parent classes from g1)
        c2: parent classes from g2)
        lexical_mappings: list of (class1, class2, score) from lexical matcher
    """
    parents1 = getParentClasses(g1, c1)
    parents2 = getParentClasses(g2, c2)

    if not parents1 or not parents2:
        return 0.0

    # Set up lookup set of lexical mappings 
    matched_pairs = set()
    for m1, m2, _ in lexical_mappings:
        matched_pairs.add((str(m1), str(m2)))
        matched_pairs.add((str(m2), str(m1)))

    # Check whether patent class is matched
    for p1 in parents1:
        for p2 in parents2:
            if (str(p1), str(p2)) in matched_pairs:
                return 1.0  

    return 0.0  

def structuralMatcher(g1, g2,lexical_threshold=0.9,combined_threshold=0.7):
    """
    Combined lexical + structural for classes.

    Scoring formula:
        combined_score = (lexical_score * 0.6) + (structural_score * 0.4)

    Weights explanation:
        Lexical:    60% weight (primary signal)
        Structural: 40% weight (supporting signal)
    """
    classes1 = getClasses(g1)
    classes2 = getClasses(g2)

    print(f"Running structural matching.")
    print(f"Classes: {len(classes1)} vs {len(classes2)}")

    # Computing progress
    lexical_mappings = []
    for c1 in classes1:
        label1 = getLabel(g1, c1)
        best_score = 0.0
        best_match = None

        for c2 in classes2:
            label2 = getLabel(g2, c2)
            score  = lexicalSimilarity(label1, label2)

            if score >= lexical_threshold and score > best_score:
                best_score = score
                best_match = c2

        if best_match is not None:
            lexical_mappings.append((c1, best_match, best_score))

    print(f"Lexical mappings found: {len(lexical_mappings)}")

    # Combined score（Lexical + Structural）
    final_mappings = []

    for c1 in classes1:
        label1 = getLabel(g1, c1)
        best_score = 0.0
        best_match = None

        for c2 in classes2:
            label2 = getLabel(g2, c2)

            # Lexical score
            lex_score  = lexicalSimilarity(label1, label2)

            # Structural score
            struct_score = structuralScore(g1, g2, c1, c2, lexical_mappings)

            # Combined score
            combined = (lex_score * 0.6) + (struct_score * 0.4)

            if combined >= combined_threshold and combined > best_score:
                best_score = combined
                best_match = c2

        if best_match is not None:
            final_mappings.append((c1, best_match,
                                   best_score))

    print(f"Final mappings found: {len(final_mappings)}")
    return final_mappings

In [29]:
# Run structural matching on cmt.owl vs ekaw.owl

start_time = time.time()

struct_mappings = structuralMatcher(g1, g2, lexical_threshold = 0.9, combined_threshold = 0.7)

end_time = time.time()
elapsed_struct = round(end_time - start_time, 2)

print(f"\nComputation time: {elapsed_struct} seconds")

# Result
print(f"\n=== Structural Matcher Results ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)

for src, tgt, score in struct_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

Running structural matching.
Classes: 29 vs 73
Lexical mappings found: 8
Final mappings found: 8

Computation time: 0.31 seconds

=== Structural Matcher Results ===
Source                              Target                              Score
---------------------------------------------------------------------------
Meta-Reviewer                       Neutral_Review                      0.755
ConferenceMember                    Conference_Participant              0.851
ExternalReviewer                    Possible_Reviewer                   0.767
Review                              Review                              1.000
Paper                               Paper                               1.000
PaperAbstract                       Poster_Paper                        0.830
PaperFullVersion                    Conference_Paper                    0.814
Meta-Review                         Neutral_Review                      0.785


In [30]:
# Section 2: Merge lexical + structural mappings
def mergeMapping(lexical_mappings, structural_mappings):
    """
    Merge lexical and structural mappings.
    
    Strategy:
    - Start with lexical mappings as base and add structural mappings that found NEW entities not already covered by lexical matcher
    - If same entity mapped differently, keep higher score
    
    """
    merged = {}

    # lexical mappings
    for c1, c2, score in lexical_mappings:
        merged[str(c1)] = (c1, c2, score)

    # structural mappings
    for c1, c2, score in structural_mappings:
        key = str(c1)

        if key not in merged:
            merged[key] = (c1, c2, score)
        else:
            existing_score = merged[key][2]
            if score > existing_score:
                merged[key] = (c1, c2, score)

    result = list(merged.values())
    print(f"Lexical mappings:    {len(lexical_mappings)}")
    print(f"Structural mappings: {len(structural_mappings)}")
    print(f"Merged mappings:     {len(result)}")
    return result

In [31]:
# Section 3: Run Merge lexical + structural mappings
lexical_class_mappings = lexicalMatcherOneToOne(g1, g2, threshold=0.9)

# Merge
print("=== Merging Lexical + Structural ===")
merged_class_mappings = mergeMapping(lexical_class_mappings,struct_mappings)

print(f"\n=== Merged Mappings Detail ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in merged_class_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9
=== Merging Lexical + Structural ===
Lexical mappings:    8
Structural mappings: 8
Merged mappings:     13

=== Merged Mappings Detail ===
Source                              Target                              Score
---------------------------------------------------------------------------
Reviewer                            Review                              0.957
Person                              Person                              1.000
Document                            Document                            1.000
Conference                          Conference                          1.000
ConferenceChair                     Conference                          0.940
ConferenceMember                    Conference                          0.931
Review                              Review                              1.000
Paper                               Paper                               1.000
Meta-Review

In [32]:
# Section 4: Evaluate: Lexical + Structural (merged)
# Reference alignment
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment
merged_file = os.path.join(output_dir, "mloa-cmt-ekaw-merged.ttl")

# Run evaluation
precision, recall, f_score = evaluate(reference_file, merged_file)

Evaluating: mloa-cmt-ekaw-merged.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 15
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.250
Recall:    0.455
F-score:   0.323


In [33]:
# Section 5: Result tracking (Lexical + Structural)

results["Task"].append("cmt-ekaw")
results["Technique"].append("Lexical + Structural (merged)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed_struct, 3))

# Display table
df_results = pd.DataFrame(results)
print(f"=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.140
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.310


### Subtask OA.1 (improve scalability)

In [34]:
# Section 1: Scalability technique- Inverted Index
# Tokenise all labels to words and only compare classes sharing common tokens

In [35]:
# Soruce: LogMap: Logic-based and Scalable Ontology Matching. ISWC 2011 by Ernesto Jiménez-Ruiz
# Repository: https://github.com/ernestojimenezruiz/logmap-matcher

import re
from collections import defaultdict

def tokenise(label):
    """
    Split a label into tokens (individual words).
    Handles camelCase, underscores, and hyphens.
    """

    label = re.sub(r'([A-Z])', r' \1', label)

    label = label.replace('_', ' ').replace('-', ' ')

    tokens = label.lower().split()

    tokens = [t for t in tokens if len(t) > 2]

    return tokens
    
def buildIndex(g):
    """
    Build an inverted index for all classes and properties in an ontology.
    Index structure:
        token → [list of entities containing this token]
    Example:
        "paper"  → [ekaw:Paper, ekaw:Conference_Paper]
        "review" → [ekaw:Review, ekaw:Neutral_Review]
    """
    index = defaultdict(list)

    # Index classes
    for entity in getClasses(g):
        label  = getLabel(g, entity)
        tokens = tokenise(label)
        for token in tokens:
            index[token].append(entity)

    # Index object properties
    for entity in getObjectProperties(g):
        label  = getLabel(g, entity)
        tokens = tokenise(label)
        for token in tokens:
            index[token].append(entity)

    # Index data properties
    for entity in getDataProperties(g):
        label  = getLabel(g, entity)
        tokens = tokenise(label)
        for token in tokens:
            index[token].append(entity)

    return index

def getCandidates(g1, g2, index2):
    """
    Use inverted index to find candidate matches, only compare entities that share at least one common token.
    """
    candidates = defaultdict(set)
    total_candidates = 0

    all_entities1 = (list(getClasses(g1)) + list(getObjectProperties(g1)) + list(getDataProperties(g1)))

    for entity1 in all_entities1:
        label = getLabel(g1, entity1)
        tokens = tokenise(label)

        for token in tokens:
            if token in index2:
                for entity2 in index2[token]:
                    candidates[entity1].add(entity2)

        total_candidates += len(candidates[entity1])

    return candidates, total_candidates

def indexedMatcher(g1, g2, threshold=0.9):
    """
    Scalable matcher using inverted index.
    """
    # Create index
    print("Building inverted index")
    index2 = buildIndex(g2)
    print(f"Index built: {len(index2)} unique tokens")

    # Searching
    print("Finding candidates")
    candidates, total_candidates = getCandidates(g1, g2, index2)

    # Compute how many saved
    all_entities1 = (list(getClasses(g1)) + list(getObjectProperties(g1)) + list(getDataProperties(g1)))

    all_entities2 = (list(getClasses(g2)) + list(getObjectProperties(g2)) +list(getDataProperties(g2)))

    brute_force = len(all_entities1) * len(all_entities2)

    print(f"\nScalability comparison:")
    print(f"  Brute force comparisons: {brute_force:,}")
    print(f"  Index-based comparisons: {total_candidates:,}")
    if brute_force > 0:
        reduction = (1 - total_candidates/brute_force) * 100
        print(f"  Reduction: {reduction:.1f}%")

    # Compare chosen pairs only
    print(f"\nMatching candidates")
    mappings = []

    for entity1, entity2_candidates in candidates.items():
        label1 = getLabel(g1, entity1)
        best_score = 0.0
        best_match = None

        for entity2 in entity2_candidates:
            label2 = getLabel(g2, entity2)
            score = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = entity2

        if best_match is not None:
            mappings.append((entity1, best_match, best_score))

    print(f"Mappings found: {len(mappings)}")
    return mappings


In [36]:
# Test: Tokenise functinon
test_labels = ["ConferenceChair", "Conference_Paper", "hasAuthor","writtenBy", "Meta-Reviewer", "Paper"]

print("=== Tokenise Test ===")
for label in test_labels:
    tokens = tokenise(label)
    print(f"  {label:<25} → {tokens}")

=== Tokenise Test ===
  ConferenceChair           → ['conference', 'chair']
  Conference_Paper          → ['conference', 'paper']
  hasAuthor                 → ['has', 'author']
  writtenBy                 → ['written']
  Meta-Reviewer             → ['meta', 'reviewer']
  Paper                     → ['paper']


In [37]:
# Section 2: Test Inverted Index Matcher on cmt.owl vs ekaw.owl
print("=" * 50)
print("BRUTE FORCE MATCHER")
print("=" * 50)
start = time.time()
bf_class = lexicalMatcherOneToOne(g1, g2, threshold=0.9)
bf_obj, bf_data = propertyMatcher(g1, g2, threshold=0.8)
elapsed_bf = round(time.time() - start, 3)
total_bf = (len(list(getClasses(g1))) * len(list(getClasses(g2)))) + (len(list(getObjectProperties(g1))) * len(list(getObjectProperties(g2))))
print(f"Total comparisons: {total_bf:,}")
print(f"Mappings found:    {len(bf_class) + len(bf_obj)}")
print(f"Time:              {elapsed_bf}s")

print()
print("=" * 50)
print("INDEXED MATCHER")
print("=" * 50)
start_time = time.time()
indexed_mappings = indexedMatcher(g1, g2, threshold=0.9)
elapsed_idx = round(time.time() - start_time, 3)
print(f"Time: {elapsed_idx}s")

idx_class_mappings = [(s, t, sc) for s, t, sc in indexed_mappings if (s, RDF.type, OWL.Class) in g1]
idx_obj_mappings = [(s, t, sc) for s, t, sc in indexed_mappings if (s, RDF.type, OWL.ObjectProperty) in g1]
idx_data_mappings = [(s, t, sc) for s, t, sc in indexed_mappings if (s, RDF.type, OWL.DatatypeProperty) in g1]

print(f"\nClass mappings:    {len(idx_class_mappings)}")
print(f"Object prop:       {len(idx_obj_mappings)}")
print(f"Data prop:         {len(idx_data_mappings)}")

print()
print("=" * 50)
print("COMPARISON")
print("=" * 50)
if elapsed_bf > 0:
    speedup = round(elapsed_bf / elapsed_idx, 1) \
              if elapsed_idx > 0 else "N/A"
    print(f"Brute force time:  {elapsed_bf}s")
    print(f"Indexed time:      {elapsed_idx}s")
    print(f"Speed improvement: {speedup}x faster")

BRUTE FORCE MATCHER
Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9
Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Total comparisons: 3,734
Mappings found:    15
Time:              0.257s

INDEXED MATCHER
Building inverted index
Index built: 85 unique tokens
Finding candidates

Scalability comparison:
  Brute force comparisons: 9,328
  Index-based comparisons: 539
  Reduction: 94.2%

Matching candidates
Mappings found: 16
Time: 0.044s

Class mappings:    9
Object prop:       5
Data prop:         2

COMPARISON
Brute force time:  0.257s
Indexed time:      0.044s
Speed improvement: 5.8x faster


In [38]:
# Section 3:The evaluation metrics of the Indexed Matcher
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment
indexed_file = os.path.join(output_dir, "mloa-cmt-ekaw-indexed.ttl")

# Run evaluation
precision, recall, f_score = evaluate(reference_file, indexed_file)

Evaluating: mloa-cmt-ekaw-indexed.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 11
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.312
Recall:    0.455
F-score:   0.370


In [39]:
# Section 4: Results Tracking (Indexed Matcher result)
results["Task"].append("cmt-ekaw")
results["Technique"].append("Indexed Matcher")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed_idx, 3))

# Display table
df_results = pd.DataFrame(results)
print(f"=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.140
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.310
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.044


### Subtask OA.1 (advanced alignment techniques - Embedding & LLM)

#### Advanced technique 1: Embedding Matching

In [40]:
# Section1: Embedding is to capture semantic similarity between ontology labels rather then just doing pure string matching between entities

In [41]:
# Model: all-MiniLM-L6-v2 (sentence-transformers)

In [42]:
# Source: Sentence Embeddings using Siamese BERT-Networks (EMNLP 2019) by Nils Reimers, Iryna Gurevych
# Repository: https://arxiv.org/abs/1908.10084
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Install SentenceTransformer(
print("Loading embedding model")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(" Model loaded")

# Create function
def getEmbedding(labels):
    """
    Convert a list of labels into embedding vectors.
    """
    return model.encode(labels)

def embeddingSimilarity(embedding1, embedding2):
    """
    Calculate cosine similarity between two embeddings.
    Measures the angle between two vector, Score = 1.0 is same meaning; Score = 0.0 is unrelated
    """
    sim = cosine_similarity(embedding1.reshape(1, -1), embedding2.reshape(1, -1))
    return float(sim[0][0])
    
def embeddingMatcher(g1, g2, threshold=0.85):
    """
    Match ontology entities using semantic embeddings.

    Strategy:
    1. Extract all labels from both ontologies
    2. Compute embeddings for all labels at once (efficient)
    3. Compare embeddings using cosine similarity
    4. Return mappings above threshold
    
    Uses one-to-one mapping constraint.
    
    Parameters:
        g1 : rdflib Graph (source ontology)
        g2 : rdflib Graph (target ontology)
        threshold : minimum cosine similarity (default 0.85)
    """
    # Class Properties
    print("=== Embedding Matcher ===")
    classes1 = getClasses(g1)
    classes2 = getClasses(g2)
    labels1 = [getLabel(g1, c) for c in classes1]
    labels2 = [getLabel(g2, c) for c in classes2]

    print(f"Computing embeddings for {len(labels1)} + "f"{len(labels2)} class labels...")

    # Calculate embeddings at once
    embeddings1 = getEmbedding(labels1)
    embeddings2 = getEmbedding(labels2)

    print(f" Embeddings computed")

    # Compare all the class pairs
    class_mappings = []
    for i, c1 in enumerate(classes1):
        best_score = 0.0
        best_match = None

        for j, c2 in enumerate(classes2):
            score = embeddingSimilarity(embeddings1[i], embeddings2[j])

            if score >= threshold and score > best_score:
                best_score = score
                best_match = c2

        if best_match is not None:
            class_mappings.append((c1, best_match, best_score))

    print(f"Class mappings found: {len(class_mappings)}")

    # Object Properties
    obj_props1 = list(getObjectProperties(g1))
    obj_props2 = list(getObjectProperties(g2))

    obj_labels1 = [getLabel(g1, p) for p in obj_props1]
    obj_labels2 = [getLabel(g2, p) for p in obj_props2]

    print(f"\nComputing embeddings for "f"{len(obj_labels1)} + {len(obj_labels2)} "f"property labels...")

    obj_emb1 = getEmbedding(obj_labels1)
    obj_emb2 = getEmbedding(obj_labels2)

    obj_mappings = []
    for i, p1 in enumerate(obj_props1):
        best_score = 0.0
        best_match = None

        for j, p2 in enumerate(obj_props2):
            score = embeddingSimilarity(
                obj_emb1[i], obj_emb2[j]
            )

            if score >= threshold and score > best_score:
                best_score = score
                best_match = p2

        if best_match is not None:
            obj_mappings.append((p1, best_match,
                                 best_score))

    print(f"Object property mappings: {len(obj_mappings)}")

    return class_mappings, obj_mappings, []

Loading embedding model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Model loaded


In [43]:
# Section2: Run Embedding matching on cmt.owl vs ekaw.owl

In [44]:
start_time = time.time()

emb_class, emb_obj, emb_data = embeddingMatcher(g1, g2, threshold=0.85)

elapsed_emb = round(time.time() - start_time, 2)
print(f"\nComputation time: {elapsed_emb}s")

# Display Class mappings
print(f"\n=== Embedding Class Mappings ===")
print(f"{'Source':<30} {'Target':<30} {'Score'}")
print("-" * 65)
for src, tgt, score in emb_class:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<30} {tgt_label:<30} {score:.3f}")

# Display property mappings
print(f"\n=== Embedding Property Mappings ===")
print(f"{'Source':<30} {'Target':<30} {'Score'}")
print("-" * 65)
for src, tgt, score in emb_obj:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<30} {tgt_label:<30} {score:.3f}")

=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 5

Computing embeddings for 49 + 33 property labels...
Object property mappings: 1

Computation time: 1.43s

=== Embedding Class Mappings ===
Source                         Target                         Score
-----------------------------------------------------------------
Person                         Person                         1.000
Document                       Document                       1.000
Conference                     Conference                     1.000
Review                         Review                         1.000
Paper                          Paper                          1.000

=== Embedding Property Mappings ===
Source                         Target                         Score
-----------------------------------------------------------------
writtenBy                      writtenBy                      1.000


In [45]:
# Try different thresholds for embedding matcher
for threshold in [0.85, 0.80, 0.75, 0.70]:
    print(f"\nThreshold: {threshold}")
    emb_c, emb_o, _ = embeddingMatcher(g1, g2, threshold=threshold)
    print(f"  Class mappings:    {len(emb_c)}")
    print(f"  Property mappings: {len(emb_o)}")
    print(f"  Total:             {len(emb_c) + len(emb_o)}")


Threshold: 0.85
=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 5

Computing embeddings for 49 + 33 property labels...
Object property mappings: 1
  Class mappings:    5
  Property mappings: 1
  Total:             6

Threshold: 0.8
=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 6

Computing embeddings for 49 + 33 property labels...
Object property mappings: 1
  Class mappings:    6
  Property mappings: 1
  Total:             7

Threshold: 0.75
=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 7

Computing embeddings for 49 + 33 property labels...
Object property mappings: 1
  Class mappings:    7
  Property mappings: 1
  Total:             8

Threshold: 0.7
=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 10

Co

In [46]:
# Section 3: Evaluate: Embedding with different thresholds to test the accuracy
for threshold in [0.85, 0.80, 0.75, 0.70]:
    # Run matcher
    emb_c, emb_o, emb_d = embeddingMatcher(g1, g2, threshold=threshold)
    
    # Save mappings
    emb_output = os.path.join(output_dir,f"mloa-cmt-ekaw-embedding-{threshold}.ttl")
    saveMappings(emb_c, emb_o, emb_d, emb_output)
    
    # Evaluate
    print(f"\n=== Threshold {threshold} ===")
    p, r, f = evaluate(reference_file, emb_output)
    print(f"Classes: {len(emb_c)}, Props: {len(emb_o)}")

=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 5

Computing embeddings for 49 + 33 property labels...
Object property mappings: 1
 Alignment saved to: C:\Users\siuju/Desktop\MLOA\Ontology_Alignments\OA.4\mloa-cmt-ekaw-embedding-0.85.ttl
 Class mappings:         5
 Object prop mappings:   1
 Data prop mappings:     0
 Total mappings:         6

=== Threshold 0.85 ===
Evaluating: mloa-cmt-ekaw-embedding-0.85.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 1
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.833
Recall:    0.455
F-score:   0.588
Classes: 5, Props: 1
=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 6

Computing embeddings for 49 + 33 property labels...
Object property mappings: 1
 Alignment saved to: C:\Users\siuju/Desktop\MLOA\Onto

In [47]:
# Section 4: Evaluate the embedding matcher (threshold=0.85)

# Reference alignment
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment
emb_file = os.path.join(output_dir, "mloa-cmt-ekaw-embedding-0.85.ttl")

# Run evaluation
precision, recall, f_score = evaluate(reference_file, emb_file)

Evaluating: mloa-cmt-ekaw-embedding-0.85.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 1
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.833
Recall:    0.455
F-score:   0.588


In [48]:
# Scetion 5: Results Tracking (Embedding Matching result)
results["Task"].append("cmt-ekaw")
results["Technique"].append("Embedding (threshold=0.85)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed_emb, 3))

# Display table
df_results = pd.DataFrame(results)
print(f"=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.140
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.310
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.044
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    1.430


In [49]:
# From the above result, we can observe that the precision of the result in embedding is higher than the previous methods.
# That means there are few incorrect mappings (low false positives) but the value of recall is keeping low. 
# To understand why, we perform a false negative analysis by comparing our system output against the reference alignment. 
# This reveals which correct mappings our system missed and why.

from rdflib import Graph

# load reference alignment
ref = Graph()
ref.parse(reference_file, format="turtle")

emb_best = Graph()
emb_best.parse(os.path.join(output_dir, "mloa-cmt-ekaw-embedding-0.85.ttl"),format="turtle")

print("=== Mappings in Reference but Not in Embedding ===")
print("False Negatives\n")
for s, p, o in ref:
    if (s, p, o) not in emb_best and (o, p, s) not in emb_best:
        s_label = str(s).split("#")[-1].split("/")[-1]
        o_label = str(o).split("#")[-1].split("/")[-1]
        print(f"  {s_label:<30} → {o_label}")
print()

print("=== Mappings in Embedding but Not in Reference ===")
print("False Positives\n")

for s, p, o in emb_best:
    if (s, p, o) not in ref and (o, p, s) not in ref:
        s_label = str(s).split("#")[-1].split("/")[-1]
        o_label = str(o).split("#")[-1].split("/")[-1]
        print(f"  {s_label:<30} → {o_label}")

=== Mappings in Reference but Not in Embedding ===
False Negatives

  Author                         → Paper_Author
  hasBeenAssigned                → reviewerOfPaper
  writtenBy                      → reviewWrittenBy
  PaperFullVersion               → Regular_Paper
  assignedTo                     → hasReviewer
  ConferenceMember               → Conference_Participant

=== Mappings in Embedding but Not in Reference ===
False Positives

  writtenBy                      → writtenBy


#### Advanced technique 2: LLM-based Matcher

In [50]:
## Section 1: LLM as Oracle for Ontology Alignment
## Sources: Large Language Models as Oracles for Ontology Alignment by Sviatoslav Lushnei1 et al.
## Repository:https://arxiv.org/pdf/2508.08500

In [51]:
## Access API: openrouter.key 
import os

def loadAPIKey():
    """
    Load OpenRouter API key from local file.
    File is stored outside the notebook directory for security.
    """
    key_path = os.path.join(os.path.expanduser("~/Desktop/MLOA"),"files", "openrouter.key.txt")
    
    print(f"Looking for key at:")
    print(f"  {key_path}")
    print()
    
    if os.path.exists(key_path):
        with open(key_path, 'r') as f:
            key = f.read().strip()
        
        if key:
            print(f" API key loaded successfully!")
            masked = key[:8] + "..." + key[-4:]
            print(f"  Key preview: {masked}")
            return key
        else:
            raise ValueError(
                "Key file is empty!\n"
                "Please put your API key inside the file."
            )
    else:
        raise FileNotFoundError(
            f"Key file not found at:\n"
            f"  {key_path}\n"
            f"Please check the file location."
        )
try:
    api_key = loadAPIKey()
except Exception as e:
    print(f" The key is missing, Error: {e}")

Looking for key at:
  C:\Users\siuju/Desktop/MLOA\files\openrouter.key.txt

 API key loaded successfully!
  Key preview: sk-or-v1...0224


In [52]:
# Section2: Create LLM function with API
# Model: nvidia/nemotron-3-super-120b-a12b:free
# Code resue and make reference to LogMapLLM 
def askLLM(label1, label2, context="conference management", api_key=None):
    """
    Ask LLM whether two ontology labels represent the same concept.
    Parameters:
        label1  : string label from ontology 1
        label2  : string label from ontology 2
        context : domain context for better understanding
        api_key : OpenRouter API key
    Returns:
        True  = LLM thinks they are equivalent
        False = LLM thinks they are Not equivalent
    """
# Check the exact match before asking LLM
    if label1.lower() == label2.lower():
        return True

    prompt = f"""Task: Ontology alignment for {context} systems.

    Are "{label1}" and "{label2}" the same concept?

    Example of same concepts:
    - "Paper" and "Article" → yes (both mean academic paper)
    - "Author" and "Writer" → yes (both mean person who writes)

    Example of different concepts:
    - "Reviewer" and "Review" → no (person vs document)
    - "Conference" and "Paper" → no (event vs document)
    
    Answer with only 'yes' or 'no'."""

    # Call API
    headers = {"Authorization": f"Bearer {api_key}","Content-Type": "application/json"}

    body = {
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 10,    
        "temperature": 0.0   
    }

    try:
        response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=body, timeout=30)

        if response.status_code == 200:
            result = response.json()
            answer = result["choices"][0]["message"]["content"].strip().lower()

            if "yes" in answer:
                return True
            else:
                return False
        else:
            print(f"API Error: {response.status_code}")
            return False

    except Exception as e:
        print(f"Request Error: {e}")
        return False

def llmMatcher(g1, g2, api_key, context="conference management", embedding_threshold=0.6, max_candidates=50):
    """
    LLM-based ontology matcher.

    Strategy:
    1. Use embedding to find candidates (low threshold)
    2. Ask LLM to verify each candidate
    3. Return verified mappings
    """
    print("=== LLM Matcher ===")
    print(f"Step 1: Finding candidates with "
          f"the lowest embedding threshold={embedding_threshold}")

    emb_candidates_c, emb_candidates_o, _ = embeddingMatcher(g1, g2,threshold=embedding_threshold)

    all_candidates = emb_candidates_c + emb_candidates_o
    print(f"Candidates found: {len(all_candidates)}")

    # Control the amount to LLm to save cost 
    if len(all_candidates) > max_candidates:
        print(f"Limiting to top {max_candidates} candidates")
        # Sorted by the embedding score 
        all_candidates = sorted(all_candidates, key=lambda x: x[2],reverse=True)[:max_candidates]

    print(f"\nStep 2: Asking LLM to verify {len(all_candidates)} candidates")
    print(f"Domain context: '{context}'\n")

    verified_mappings = []
    rejected_mappings = []

    for i, (e1, e2, emb_score) in enumerate(all_candidates):

        label1 = getLabel(g1, e1)
        label2 = getLabel(g2, e2)

        is_equivalent = askLLM(label1, label2, context=context, api_key=api_key)

        if is_equivalent:
            verified_mappings.append((e1, e2, emb_score))
            status = "YES"
        else:
            rejected_mappings.append((e1, e2, emb_score))
            status = "NO"

        print(f"  [{i+1}/{len(all_candidates)}] "
              f"{label1:<25} → {label2:<25} {status}")

        time.sleep(0.5)

    print(f"\nStep 3: Results")
    print(f"  Verified:  {len(verified_mappings)}")
    print(f"  Rejected:  {len(rejected_mappings)}")

    return verified_mappings

In [53]:
# Section3: Run LLM Matcher on cmt.owl vs ekaw.owl

start_time = time.time()

llm_mappings = llmMatcher(g1, g2, api_key=api_key, context="conference management and paper review system", embedding_threshold=0.6, max_candidates=50)

elapsed_llm = round(time.time() - start_time, 2)
print(f"\nTotal time: {elapsed_llm}s")

llm_class_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.Class) in g1]
llm_obj_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.ObjectProperty) in g1]

print(f"\n=== Mapping Summary ===")
print(f"Class mappings:    {len(class_mappings)}")
print(f"Object prop:       {len(obj_mappings)}")
print(f"Data prop:         {len(data_mappings)}")
print(f"Total:             "
      f"{len(class_mappings) + len(obj_mappings) + len(data_mappings)}")

=== LLM Matcher ===
Step 1: Finding candidates with the lowest embedding threshold=0.6
=== Embedding Matcher ===
Computing embeddings for 29 + 73 class labels...
 Embeddings computed
Class mappings found: 14

Computing embeddings for 49 + 33 property labels...
Object property mappings: 14
Candidates found: 28

Step 2: Asking LLM to verify 28 candidates
Domain context: 'conference management and paper review system'

  [1/28] Meta-Reviewer             → Possible_Reviewer         NO
  [2/28] Reviewer                  → Review                    YES
  [3/28] Person                    → Person                    YES
  [4/28] Document                  → Document                  YES
  [5/28] Conference                → Conference                YES
  [6/28] ConferenceChair           → Conference                YES
  [7/28] ConferenceMember          → Conference                YES
  [8/28] Review                    → Review                    YES
  [9/28] Paper                     → Paper   

In [54]:
# Section 4: Evaluate: LLM Matcher

llm_class_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.Class) in g1]
llm_obj_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.ObjectProperty) in g1]

print(f"LLM Class mappings:    {len(llm_class_mappings)}")
print(f"LLM Property mappings: {len(llm_obj_mappings)}")

llm_output = os.path.join(output_dir, "mloa-cmt-ekaw-llm.ttl")
saveMappings(llm_class_mappings, llm_obj_mappings, [], llm_output)

print()

reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")
precision, recall, f_score = evaluate(reference_file, llm_output)

LLM Class mappings:    10
LLM Property mappings: 2
 Alignment saved to: C:\Users\siuju/Desktop\MLOA\Ontology_Alignments\OA.4\mloa-cmt-ekaw-llm.ttl
 Class mappings:         10
 Object prop mappings:   2
 Data prop mappings:     0
 Total mappings:         12

Evaluating: mloa-cmt-ekaw-llm.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 7
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.417
Recall:    0.455
F-score:   0.435


In [55]:
#  Section 5:Results Tracking: Add LLM Matcher result
results["Task"].append("cmt-ekaw")
results["Technique"].append("LLM (NVIDIA Nemotron)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(92.44)

# Display table
df_results = pd.DataFrame(results)
print(f"\n=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")


=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.140
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.310
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.044
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    1.430
cmt-ekaw     LLM (NVIDIA Nemotron)                    0.417    0.455    0.435   92.440


#### Advanced technique 3: Merge Embedding and LLM-based Matcher

In [56]:
# Section 1:
print("=== Merging Embedding + LLM Mappings ===")

# Combine class mappings
final_class = mergeMapping(
    emb_class, # Embedding class mappings
    llm_class_mappings # LLM class mappings
)

# Combine property mappings
final_obj = mergeMapping(
    emb_obj, # Embedding property mappings
    llm_obj_mappings # LLM property mappings
)

print(f"\nEmbedding class mappings:  {len(emb_class)}")
print(f"LLM class mappings:        {len(llm_class_mappings)}")
print(f"Merged class mappings:     {len(final_class)}")
print(f"\nEmbedding obj mappings:    {len(emb_obj)}")
print(f"LLM obj mappings:          {len(llm_obj_mappings)}")
print(f"Merged obj mappings:       {len(final_obj)}")

# 
final_output = os.path.join(output_dir, "mloa-cmt-ekaw-final.ttl")
saveMappings(final_class, final_obj, [], final_output)
print(f"\n Embedding + LLM alignment saved!")

=== Merging Embedding + LLM Mappings ===
Lexical mappings:    5
Structural mappings: 10
Merged mappings:     10
Lexical mappings:    1
Structural mappings: 2
Merged mappings:     2

Embedding class mappings:  5
LLM class mappings:        10
Merged class mappings:     10

Embedding obj mappings:    1
LLM obj mappings:          2
Merged obj mappings:       2
 Alignment saved to: C:\Users\siuju/Desktop\MLOA\Ontology_Alignments\OA.4\mloa-cmt-ekaw-final.ttl
 Class mappings:         10
 Object prop mappings:   2
 Data prop mappings:     0
 Total mappings:         12

 Embedding + LLM alignment saved!


In [57]:
# Section 2: Evaluate the system of Embedding + LLM alignment
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

precision, recall, f_score = evaluate(reference_file, final_output)

Evaluating: mloa-cmt-ekaw-final.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 7
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.417
Recall:    0.455
F-score:   0.435


In [58]:
# Section 3:Results Tracking (Embedding + LLM alignmen)

results["Task"].append("cmt-ekaw")
results["Technique"].append("Embedding + LLM")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(1.34 + 92.44, 2))

# Display table
df_results = pd.DataFrame(results)
print(f"\n=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")


=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.140
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.310
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.044
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    1.430
cmt-ekaw     LLM (NVIDIA Nemotron)                    0.417    0.455    0.435   92.440
cmt-ekaw     Embedding + LLM                          0.417    0.455    0.435   93.780


In [59]:
# Section 4: Check the difference between Embedding and LLM mapping 

print("=== Embedding Class Mappings ===")
for s, t, sc in emb_class:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

print(f"\n=== LLM Class Mappings ===")
for s, t, sc in llm_class_mappings:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

print(f"\n=== Merged Class Mappings ===")
for s, t, sc in final_class:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

=== Embedding Class Mappings ===
  Person                         → Person                         1.000
  Document                       → Document                       1.000
  Conference                     → Conference                     1.000
  Review                         → Review                         1.000
  Paper                          → Paper                          1.000

=== LLM Class Mappings ===
  Reviewer                       → Review                         0.810
  Person                         → Person                         1.000
  Document                       → Document                       1.000
  Conference                     → Conference                     1.000
  ConferenceChair                → Conference                     0.659
  ConferenceMember               → Conference                     0.740
  Review                         → Review                         1.000
  Paper                          → Paper                          1.000
  P

In [60]:
# Section 5:Filter LLM mapping with embedding score >=0.75

LLM_SCORE_THRESHOLD = 0.75

# Filter low score 
filtered_llm_class = [
    (s, t, sc) for s, t, sc in llm_class_mappings
    if sc >= LLM_SCORE_THRESHOLD
]

filtered_llm_obj = [
    (s, t, sc) for s, t, sc in llm_obj_mappings
    if sc >= LLM_SCORE_THRESHOLD
]

print(f"=== LLM Mappings Filtering ===")
print(f"Original LLM class mappings:  {len(llm_class_mappings)}")
print(f"Filtered LLM class mappings:  {len(filtered_llm_class)}")
print(f"\nKept mappings:")
for s, t, sc in filtered_llm_class:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

print(f"\nRemoved mappings:")
removed = [
    (s, t, sc) for s, t, sc in llm_class_mappings
    if sc < LLM_SCORE_THRESHOLD
]
for s, t, sc in removed:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

=== LLM Mappings Filtering ===
Original LLM class mappings:  10
Filtered LLM class mappings:  7

Kept mappings:
  Reviewer                       → Review                         0.810
  Person                         → Person                         1.000
  Document                       → Document                       1.000
  Conference                     → Conference                     1.000
  Review                         → Review                         1.000
  Paper                          → Paper                          1.000
  PaperAbstract                  → Paper                          0.756

Removed mappings:
  ConferenceChair                → Conference                     0.659
  ConferenceMember               → Conference                     0.740
  Meta-Review                    → Review                         0.700


In [61]:
# Section 6: Evaluate Embedding + Filtered LLM (Final)

# Merge Embedding + Filtered LLM
final_class_filtered = mergeMapping(emb_class, filtered_llm_class)
final_obj_filtered = mergeMapping(emb_obj, filtered_llm_obj)

print(f"Final class mappings:  {len(final_class_filtered)}")
print(f"Final obj mappings:    {len(final_obj_filtered)}")


final_filtered_output = os.path.join(output_dir, "mloa-cmt-ekaw-final-filtered.ttl")
saveMappings(final_class_filtered, final_obj_filtered, [], final_filtered_output)

print()


reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")
precision, recall, f_score = evaluate(reference_file, final_filtered_output)

Lexical mappings:    5
Structural mappings: 7
Merged mappings:     7
Lexical mappings:    1
Structural mappings: 1
Merged mappings:     1
Final class mappings:  7
Final obj mappings:    1
 Alignment saved to: C:\Users\siuju/Desktop\MLOA\Ontology_Alignments\OA.4\mloa-cmt-ekaw-final-filtered.ttl
 Class mappings:         7
 Object prop mappings:   1
 Data prop mappings:     0
 Total mappings:         8

Evaluating: mloa-cmt-ekaw-final-filtered.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 3
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.625
Recall:    0.455
F-score:   0.526


In [62]:
# Section 7: Results Tracking 
results["Task"].append("cmt-ekaw")
results["Technique"].append("Embedding + LLM Filtered (0.75)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(1.38 + 92.44, 2))

df_results = pd.DataFrame(results)
print(f"\n=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")


=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.140
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.310
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.044
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    1.430
cmt-ekaw     LLM (NVIDIA Nemotron)                    0.417    0.455    0.435   92.440
cmt-ekaw     Embedding + LLM                          0.417    0.455    0.435   93.780
cmt-ekaw     Embedding + LLM Filtered (0.75)          0.625    0.455    0.526   93.820
